# BERT Multi Classification 모델 구현 실습 - PyTorch

이 노트북은 뉴스 데이터를 이용한 **BERT 기반 다중 분류 모델**을 PyTorch 코드로 구현합니다.

## PDF 내용 요약

- 뉴스 데이터 세트를 Pandas로 읽고, `news.csv`를 사용하여 기사 본문과 카테고리를 준비합니다.
- `category.value_counts()`로 클래스별 데이터 수를 확인하고, Train/Validation/Test 데이터 크기를 확인합니다.
- 뉴스 본문(`news`)과 라벨(`category`)을 사용하는 Dataset 클래스를 정의합니다.
- `kykim/bert-kor-base` Pre-trained 모델을 다운로드하고, `num_labels=8`처럼 분류할 클래스 수를 지정합니다. 이후 Epoch, Batch Size, Weight Decay 등의 하이퍼파라미터를 설정합니다.


## 1. 패키지 설치

뉴스 다중 분류도 BERT Tokenizer와 BERT 분류 모델을 사용하므로 `transformers`, `accelerate`, `scikit-learn`을 설치합니다.

In [59]:
# Colab 환경에서 필요한 패키지를 설치합니다.
# transformers: Hugging Face의 BERT 모델과 Tokenizer를 사용하기 위한 라이브러리입니다.
# accelerate: PyTorch 학습 장치 설정을 보조하는 라이브러리로, 최신 transformers와 함께 자주 사용됩니다.
# scikit-learn: 데이터 분리와 평가 지표 계산에 사용합니다.
!pip -q install transformers accelerate scikit-learn

## 2. 라이브러리 불러오기와 재현성 설정

Binary Classification과 동일하게 PyTorch, Pandas, Hugging Face Transformers, Scikit-learn을 사용합니다.

In [60]:
# 운영체제 경로 처리와 파일 확인에 사용하는 표준 라이브러리입니다.
import os

# 난수 시드 고정을 위해 사용하는 표준 라이브러리입니다.
import random

# 배열 연산과 난수 제어를 위해 NumPy를 불러옵니다.
import numpy as np

# CSV 파일을 읽고 표 형태 데이터를 처리하기 위해 Pandas를 불러옵니다.
import pandas as pd

# PyTorch Tensor, 모델, 학습 연산을 사용하기 위해 torch를 불러옵니다.
import torch

# PyTorch Dataset과 DataLoader를 사용하기 위해 필요한 클래스를 불러옵니다.
from torch.utils.data import Dataset, DataLoader

# 데이터셋을 Train/Validation/Test로 나누기 위해 train_test_split을 불러옵니다.
from sklearn.model_selection import train_test_split

# 정확도와 상세 분류 리포트를 계산하기 위해 평가 함수를 불러옵니다.
from sklearn.metrics import accuracy_score, classification_report

# 한국어 BERT Tokenizer를 불러오기 위한 클래스입니다.
from transformers import BertTokenizerFast

# 문장 다중 분류용 BERT 모델을 불러오기 위한 클래스입니다.
from transformers import BertForSequenceClassification

# Transformer 학습에 적합한 AdamW Optimizer를 불러옵니다.
from torch.optim import AdamW

# 학습률 스케줄러를 불러옵니다.
from transformers import get_linear_schedule_with_warmup

# 반복문 진행률을 표시하기 위해 tqdm을 불러옵니다.
from tqdm.auto import tqdm

# 실험 재현성을 위한 난수 시드를 지정합니다.
SEED = 42

# 파이썬 random 모듈의 난수 시드를 고정합니다.
random.seed(SEED)

# NumPy 난수 시드를 고정합니다.
np.random.seed(SEED)

# PyTorch CPU 난수 시드를 고정합니다.
torch.manual_seed(SEED)

# GPU가 있으면 CUDA 난수 시드도 고정합니다.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU가 사용 가능하면 cuda, 아니면 cpu를 학습 장치로 설정합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 학습 장치를 출력합니다.
print("사용 장치:", device)

사용 장치: cpu


## 3. 뉴스 데이터 로드

`news.csv`를 Pandas로 읽어 뉴스 기사와 카테고리 데이터를 준비합니다.

In [61]:
# # Colab에서 직접 파일 업로드를 지원하기 위해 try 문을 사용합니다.
try:
#     # Colab 파일 업로드 기능을 불러옵니다.
    from google.colab import files
    from google.colab import drive
    drive.mount('/content/drive')

# # Colab이 아닌 환경에서는 google.colab 모듈이 없을 수 있습니다.
except Exception:
#     # Colab이 아니면 files 변수를 None으로 둡니다.
    files = None

# # 기본 뉴스 데이터 경로를 지정합니다.
# DATA_PATH = "./data/news.csv"
DATA_PATH = '/content/drive/MyDrive/data/news2.csv'

# # 현재 폴더에 news.csv가 있을 때 사용할 대체 경로입니다.
LOCAL_FALLBACK_PATH = "./news2.csv"


# ./data/news.csv 파일이 있으면 해당 경로를 사용합니다.
if os.path.exists(DATA_PATH):
    data_path = DATA_PATH

# ./news.csv 파일이 있으면 해당 경로를 사용합니다.
elif os.path.exists(LOCAL_FALLBACK_PATH):
    data_path = LOCAL_FALLBACK_PATH

# 파일이 없고 Colab 업로드 기능을 사용할 수 있으면 업로드를 요청합니다.
elif files is not None:
    # 사용자에게 news.csv 파일 업로드 창을 띄웁니다.
    uploaded = files.upload()

    # 업로드된 첫 번째 파일명을 가져옵니다.
    uploaded_name = next(iter(uploaded.keys()))

    # 업로드된 파일명을 데이터 경로로 사용합니다.
    data_path = uploaded_name

# 모든 방법이 실패하면 파일 없음 오류를 발생시킵니다.
else:
    raise FileNotFoundError("news.csv 파일을 ./data/news.csv 또는 현재 폴더에 배치하세요.")

# 뉴스 CSV 파일을 Pandas DataFrame으로 읽습니다.
dataset = pd.read_csv(data_path)

# 데이터가 정상적으로 읽혔는지 상위 5개 행을 출력합니다.
dataset.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,주소,일자,언론사,기고자,제목,통합 분류1,통합 분류2,통합 분류3,사건_사고 분류1,사건_사고 분류2,사건_사고 분류3,개체명(인물),개체명(지역),개체명(기업기관),키워드,특성추출,본문,원본주소
0,http://www.bigkinds.or.kr/news/newsDetailView....,2021-07-23,세계일보,서필웅,양궁·태권도·펜싱 출격… 황홀한 ‘황금 주말’ 예고,스포츠>올림픽_아시안게임,스포츠>축구,스포츠>월드컵,NaN,NaN,NaN,"장민희, 이대훈, 강채영","진천, 루마니아, 맨체스터, 일본, 슬럼프, 경기, 영광, 한국, 리우데자네이루, ...","국제대, 정부, 유일, 도쿄, 금빛, 연합뉴스, 한국, 대표팀, 한국선수단","양궁,태권도,펜싱,출격,황홀,황금,주말,예고,종주국,자존심,태권,도쿄,올림픽,남자,...","금메달,한국,랭킹라운드,1위,도쿄올림픽,일본,선수들,진천,감염증,도쿄,개인전",“종주국 자존심 지킨다” 25일 도쿄올림픽 태권도 남자 68㎏급에 출전하는 이대훈(...,http://www.segye.com/content/html/2021/07/23/2...
1,http://www.bigkinds.or.kr/news/newsDetailView....,2021-07-23,국민일보,김철오,[세상에 없던 올림픽] 자위대 검색·킥보드 보안요원… 긴장감 속 분주해진 MPC,국제>일본,NaN,NaN,NaN,NaN,NaN,NaN,"제2차 세계대전, 일본, 도쿄, 경기, 고토구","정부, 유일, 일본, 프레스센터, 위대원, 신원, 자원, 자위대, 국제방송센터, I...","자위대,검색,킥보드,보안요원,긴장감,분주,긴장감,MPC,취재진,관전자,무관,중대회,...","mpc,취재진,자위대,일본,코로나19,자위대원,긴장감,도쿄올림픽,검색대,도쿄,킥보드...",도쿄올림픽 개막이 하루 앞으로 다가오자 일본 도쿄 고토구 메인프레스센터(MPC)를 ...,http://news.kmib.co.kr/article/view.asp?arcid=...
2,http://www.bigkinds.or.kr/news/newsDetailView....,2021-07-23,한겨레,이준희,"코로나 시대, ‘위로’의 올림픽 개막",문화>문화일반,NaN,NaN,사고>산업사고>화재,재해>자연재해>지진,NaN,"황선우, 김연경","신주쿠, 일본, 이와테, 도쿄, 미야기, 동일본, 대한민국, 후쿠시마, 태양","일본, 태극기, 한국, 조직위원회","코로나,시대,위로,올림픽,개막,성화,결국,올림픽,사상,초유,대회,급속도,확산,코로나...","개막식,일본,도쿄,코로나,동일본,금메달,후쿠시마,성화대,도쿄올림픽,만큼,국립경기장,...",결국 올림픽 성화에 불이 붙었다... .. .. ..사상 초유의 대회다... 지난해...,http://www.hani.co.kr/arti/sports/sportstemp/1...
3,http://www.bigkinds.or.kr/news/newsDetailView....,2021-07-23,조선일보,노석조 기자,[단독] 해외 파견 외교관 접종률 35%,국제>국제일반,NaN,NaN,NaN,NaN,NaN,태영호,"광저우, 중국, 중남미, 아프리카, 한국, 칭다오, 중동, 상하이, 우한, 도쿄, ...","외교통일위원회, 청해부대, 외교부, 국회, 국민의힘","35%,해외,파견,외교관,접종,35%,세계,공관,파견,외교관,접종률,코로나,백신,접...","접종률,코로나,중국,외교부,아프리카,외교관,스푸트니크,러시아산,도쿄,시노팜,태영호,...",전 세계 공관에 파견된 외교관의 코로나 백신 접종률이 35% 수준인 것으로 23일 ...,https://www.chosun.com/politics/politics_gener...
4,http://www.bigkinds.or.kr/news/newsDetailView....,2021-07-23,MBC,김태운,'우여곡절 도쿄2020'…사상 최초 무관중 개막식,국제>일본,국제>중남미,NaN,사고>산업사고>화재,NaN,NaN,최다인,"도쿄2020, 일본, 도쿄, 경기",일본 기상청,"우여곡절,도쿄,사상,무관,개막식,앵커,2021년,도쿄올림픽,개막,우여곡절,시작,걱정...","개막식,도쿄,경기장,일본,확진자,우여곡절,선수단,남쪽,코로나,참가자,김태운,불꽃놀이...",◀ 앵커 ..▶ .. ..2020을 달고 2021년에 열리게 된 도쿄올림픽이 잠시 ...,https://imnews.imbc.com/replay/2021/nwdesk/art...


In [62]:
dataset['소분류'] = dataset['통합 분류1'].str.split(">").str[0]
print(dataset['소분류'].value_counts())
label_map  = {
    # # '스포츠' :    7,
    # '국제'   :    6,
    # '사회'   :    5,
    # '문화'   :   4,
    # '경제'   :    3,
    # '정치'   :    2,
    # '지역'   :     1,
    # 'IT과학':     0
    '스포츠': 6,
    '국제': 5,
    '사회': 4,
    '문화': 3,
    '경제': 2,
    '정치': 1,
    '지역': 0
}


    # '지역': 0,
    # '정치': 1,
    # '경제': 2,
    # '문화': 3,
    # '사회': 4,
    # '국제': 5,
    # '스포츠': 6
    
dataset['category'] = dataset['소분류'].map(label_map)
# category가 NaN인 행 제거
dataset = dataset.dropna(subset=["category"])
dataset['category'] = dataset['category'].astype(int)

dataset['news'] = dataset['본문']

dataset = dataset[['category', 'news']]

dataset['num'] = [n + 1 for n in range(len(dataset))]

print(dataset.tail())

# 카테고리별 데이터를 200개씩으로 맞추는 코드임
balanced_dataset = (
    dataset
    .groupby("category", group_keys=False)
    .apply(lambda x: x.sample(n=200, replace=len(x) < 200, random_state=42))
    .reset_index(drop=True)
)

print(balanced_dataset["category"].value_counts().sort_index())

소분류
스포츠      9847
국제       1893
사회        346
문화        339
경제        175
정치        156
지역         95
IT_과학      58
Name: count, dtype: int64
       category                                               news    num
13090         6  [아시아경제 임주형 기자] MBC 도쿄 올림픽 중계가 선수에 대한 부적절한 발언으로...  12847
13091         6  지난달 31일 토요일 저녁때의 일이다... 도쿄올림픽에서는 축구경기(한국과 멕시코)...  12848
13092         6  박희준이 지난 6일 가라테 남자 가타 동메달 결정전에서 경기를 펼치고 있다... 연...  12849
13093         5  [앵커] ..일본의 코로나19 상황이 심상치 않습니다. .. ..신규 확진자 수가 ...  12850
13094         5  [앵커] ..코로나 대유행 속에 열린 도쿄올림픽, 17일간의 여정이 오늘 밤 끝납니...  12851
category
0    200
1    200
2    200
3    200
4    200
5    200
6    200
Name: count, dtype: int64


/tmp/ipykernel_786/157955559.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=200, replace=len(x) < 200, random_state=42))


In [63]:
# print(dataset['통합분류1'].head(), dataset['본문'].head())

## 4. 컬럼 확인, 결측치 제거, 클래스 분포 확인

클래스별 데이터 개수를 확인합니다. 다중 분류에서는 각 클래스의 데이터 수가 균형적인지 확인하는 것이 중요합니다.

In [64]:
# 데이터셋의 컬럼명을 출력하여 news, category 컬럼이 있는지 확인합니다.
print("컬럼 목록:", dataset.columns.tolist())

# category와 news 컬럼이 없으면 오류를 발생시켜 컬럼명을 확인하도록 합니다.
if "category" not in dataset.columns or "news" not in dataset.columns:
    raise ValueError("news.csv 파일에는 'category' 컬럼과 'news' 컬럼이 필요합니다.")

# 결측치 제거 전 데이터 개수를 저장합니다.
before_count = len(dataset)

# news 또는 category에 결측치가 있는 행을 제거합니다.
dataset = dataset.dropna(subset=["news", "category"]).reset_index(drop=True)

# category를 정수형으로 변환합니다.
dataset["category"] = dataset["category"].astype(int)

# 결측치 제거 후 데이터 개수를 저장합니다.
after_count = len(dataset)

# 제거 전 데이터 수를 출력합니다.
print("결측치 제거 전 데이터 수:", before_count)

# 제거 후 데이터 수를 출력합니다.
print("결측치 제거 후 데이터 수:", after_count)

# 카테고리별 데이터 개수를 출력합니다.
print(dataset["category"].value_counts().sort_index())

# 카테고리가 7인 데이터가 있으면 예시를 출력합니다.
dataset[dataset["category"] == 7].head()

컬럼 목록: ['category', 'news', 'num']
결측치 제거 전 데이터 수: 12851
결측치 제거 후 데이터 수: 12851
category
0      95
1     156
2     175
3     339
4     346
5    1893
6    9847
Name: count, dtype: int64


,category,news,num


## 5. Train / Validation / Test 데이터 분리

뉴스 카테고리 비율을 유지하기 위해 `stratify=dataset['category']`를 적용합니다.

In [65]:
# 전체 데이터에서 Train/Test 인덱스를 분리합니다.
train_idx, test_idx, _, _ = train_test_split(
    dataset.index,                 # 분리할 전체 데이터의 인덱스입니다.
    dataset["category"],           # 카테고리 비율을 유지하기 위한 기준 레이블입니다.
    test_size=0.2,                 # 전체 데이터 중 20%를 Test 데이터로 사용합니다.
    stratify=dataset["category"],  # 각 카테고리 비율을 유지합니다.
    random_state=SEED              # 재현성을 위해 난수 시드를 고정합니다.
)

# Train 인덱스에 해당하는 데이터를 선택합니다.
train_set = dataset.iloc[train_idx].reset_index(drop=True)

# Test 인덱스에 해당하는 데이터를 선택합니다.
test_set = dataset.iloc[test_idx].reset_index(drop=True)

# Train 데이터에서 다시 Train/Validation 인덱스를 분리합니다.
train_idx, valid_idx, _, _ = train_test_split(
    train_set.index,                 # 다시 분리할 Train 데이터 인덱스입니다.
    train_set["category"],           # 카테고리 비율을 유지하기 위한 기준 레이블입니다.
    test_size=0.2,                   # Train 데이터 중 20%를 Validation으로 사용합니다.
    stratify=train_set["category"],  # Train/Validation에도 카테고리 비율을 유지합니다.
    random_state=SEED                # 재현성을 위해 난수 시드를 고정합니다.
)

# Validation 데이터를 선택합니다.
valid_set = train_set.iloc[valid_idx].reset_index(drop=True)

# 최종 Train 데이터를 선택합니다.
train_set = train_set.iloc[train_idx].reset_index(drop=True)

# Train 데이터 크기를 출력합니다.
print("Train:", train_set.shape)

# Validation 데이터 크기를 출력합니다.
print("Validation:", valid_set.shape)

# Test 데이터 크기를 출력합니다.
print("Test:", test_set.shape)

Train: (8224, 3)
Validation: (2056, 3)
Test: (2571, 3)


## 6. 뉴스 다중 분류 Dataset 클래스 정의

`news`와 `category`를 필드로 사용하는 Dataset 클래스를 정의합니다. 반환 형식은 BERT 입력에 맞게 `input_ids`, `attention_mask`, `labels`입니다.

In [66]:
# 뉴스 다중 분류용 Dataset 클래스를 정의합니다.
class BertNewsDataset(Dataset):
    # Dataset 객체 생성 시 뉴스 본문, 카테고리, Tokenizer, 최대 길이를 저장합니다.
    def __init__(self, news, category, tokenizer, max_len=128):
        # 뉴스 본문 리스트를 저장합니다.
        self.news = news

        # 카테고리 레이블 리스트를 저장합니다.
        self.category = category

        # BERT Tokenizer를 저장합니다.
        self.tokenizer = tokenizer

        # 모든 문장의 최대 토큰 길이를 저장합니다.
        self.max_len = max_len

    # 전체 데이터 개수를 반환합니다.
    def __len__(self):
        # 뉴스 리스트의 길이를 반환합니다.
        return len(self.news)

    # 특정 index에 해당하는 샘플 하나를 반환합니다.
    def __getitem__(self, index):
        # index 위치의 뉴스 본문을 문자열로 변환합니다.
        text = str(self.news[index])

        # index 위치의 카테고리를 정수로 변환합니다.
        label = int(self.category[index])

        # Tokenizer로 뉴스 본문을 BERT 입력 형식으로 변환합니다.
        # encoded = self.tokenizer.encode_plus(
        encoded = self.tokenizer(
            text,                          # 토큰화할 뉴스 본문입니다.
            add_special_tokens=True,        # [CLS], [SEP] 토큰을 추가합니다.
            max_length=self.max_len,        # 최대 토큰 길이를 지정합니다.
            padding="max_length",          # 짧은 문장을 패딩합니다.
            truncation=True,                # 긴 문장을 최대 길이에 맞게 자릅니다.
            return_attention_mask=True,     # 실제 토큰과 패딩을 구분하는 mask를 반환합니다.
            return_token_type_ids=False,    # 단일 문장 분류이므로 token_type_ids는 사용하지 않습니다.
            return_tensors="pt"             # PyTorch Tensor로 반환합니다.
        )

        # 모델 입력과 정답 레이블을 딕셔너리로 반환합니다.
        return {
            "input_ids": encoded["input_ids"].squeeze(0),           # [1, max_len]을 [max_len]으로 변환합니다.
            "attention_mask": encoded["attention_mask"].squeeze(0), # attention_mask도 [max_len]으로 변환합니다.
            "labels": torch.tensor(label, dtype=torch.long)          # 다중 분류 레이블을 long Tensor로 변환합니다.
        }

## 7. Tokenizer와 Dataset 객체 생성

`kykim/bert-kor-base` Tokenizer를 다운로드한 뒤 Train/Validation/Test Dataset을 생성합니다.

In [67]:
# 사용할 한국어 BERT 모델 이름을 지정합니다.
bert_model_name = "kykim/bert-kor-base"

# Hugging Face에서 사전 학습된 BERT Tokenizer를 다운로드합니다.
tokenizer = BertTokenizerFast.from_pretrained(bert_model_name)

# BERT 입력 최대 토큰 길이를 지정합니다.
MAX_LEN = 128

# Train DataFrame을 Dataset으로 변환합니다.
train_dataset = BertNewsDataset(
    news=train_set["news"].tolist(),          # Train 뉴스 본문 리스트입니다.
    category=train_set["category"].tolist(),  # Train 카테고리 레이블 리스트입니다.
    tokenizer=tokenizer,                       # BERT Tokenizer입니다.
    max_len=MAX_LEN                            # 최대 토큰 길이입니다.
)

# Validation DataFrame을 Dataset으로 변환합니다.
valid_dataset = BertNewsDataset(
    news=valid_set["news"].tolist(),          # Validation 뉴스 본문 리스트입니다.
    category=valid_set["category"].tolist(),  # Validation 카테고리 레이블 리스트입니다.
    tokenizer=tokenizer,                       # 같은 BERT Tokenizer입니다.
    max_len=MAX_LEN                            # 같은 최대 토큰 길이입니다.
)

# Test DataFrame을 Dataset으로 변환합니다.
test_dataset = BertNewsDataset(
    news=test_set["news"].tolist(),           # Test 뉴스 본문 리스트입니다.
    category=test_set["category"].tolist(),   # Test 카테고리 레이블 리스트입니다.
    tokenizer=tokenizer,                       # 같은 BERT Tokenizer입니다.
    max_len=MAX_LEN                            # 같은 최대 토큰 길이입니다.
)

# 첫 번째 Train 샘플의 구조를 확인합니다.
print(train_dataset[0])

{'input_ids': tensor([    2, 21120, 20555, 22539,  8008, 20137,  8112, 14147, 14891,  8232,
         8123, 24547,  8018, 18319, 26187,  8061,  6169,  8837,  8575, 35179,
        15644,  3110,  9677, 17451, 22403,  8065,  8054, 19983, 13990,  2016,
         2016,  2016, 29301, 24879, 20555, 22539, 14891,  8232,  8123, 33512,
         8008, 14530,  6622, 40908, 17322, 13990,  2016,  2016,  2016, 14891,
         8232,  8123, 24547,  8078, 18319, 16049, 14339, 22315,  8095,  8215,
         5504, 29971, 13970, 26187,  8273, 19604, 19438, 35277,  8103, 21393,
         6831, 18572,  2016,  2016,  2016, 15984,  8541, 28686,  8034,  2019,
        15419, 14801,  8152, 17927,  2014, 33046,  8147, 14702,  8152, 19772,
         2014, 32749,  8147,  2020,  2016,  2016,     3,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

## 8. DataLoader 생성

뉴스 데이터를 미니배치 단위로 모델에 공급하기 위해 DataLoader를 생성합니다.

In [68]:
# GPU 메모리에 맞게 Batch Size를 지정합니다.
BATCH_SIZE = 16

# Train Dataset을 학습용 DataLoader로 변환합니다.
train_loader = DataLoader(
    train_dataset,          # 학습용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 한 번에 학습할 샘플 수입니다.
    shuffle=True            # 학습 시 데이터 순서를 섞습니다.
)

# Validation Dataset을 검증용 DataLoader로 변환합니다.
valid_loader = DataLoader(
    valid_dataset,          # 검증용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 검증용 Batch Size입니다.
    shuffle=False           # 검증 시 순서를 섞지 않습니다.
)

# Test Dataset을 평가용 DataLoader로 변환합니다.
test_loader = DataLoader(
    test_dataset,           # 평가용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 평가용 Batch Size입니다.
    shuffle=False           # 평가 시 순서를 섞지 않습니다.
)

## 9. Pre-trained BERT 모델 다운로드와 출력 클래스 수 설정

`num_labels`는 분류할 카테고리 개수입니다. 예제에서는 8개 클래스를 사용합니다.

In [69]:
# 데이터에 존재하는 고유 카테고리 개수를 계산합니다.
NUM_LABELS = dataset["category"].nunique()

# 카테고리 값이 0부터 연속적으로 구성되어 있는지 확인하기 위해 정렬된 목록을 만듭니다.
label_values = sorted(dataset["category"].unique().tolist())

# 현재 데이터의 클래스 목록을 출력합니다.
print("클래스 목록:", label_values)

# 출력층 클래스 수를 출력합니다.
print("NUM_LABELS:", NUM_LABELS)

# 다중 분류용 BERT 모델을 다운로드합니다.
model = BertForSequenceClassification.from_pretrained(
    bert_model_name,      # 사용할 사전 학습 모델 이름입니다.
    num_labels=NUM_LABELS # 다중 분류 클래스 개수입니다.
)

# 모델을 GPU 또는 CPU 장치로 이동합니다.
model = model.to(device)

클래스 목록: [0, 1, 2, 3, 4, 5, 6]
NUM_LABELS: 7


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: kykim/bert-kor-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

## 10. Fine-tuning 전략 설정

Binary Classification과 동일하게, BERT의 일부 계층을 고정하여 학습 시간과 메모리 사용량을 줄일 수 있습니다.

In [70]:
# Fine-tuning 전략을 지정합니다. 0은 전체 학습, 1은 BERT 전체 고정, 2는 pooler만 학습, 3은 마지막 encoder와 pooler만 학습입니다.
tl_strategy = 3

# 전략 1: BERT 본체 전체를 고정합니다.
if tl_strategy == 1:
    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # 현재 파라미터 이름을 출력합니다.
        print(name)

        # 해당 파라미터를 학습하지 않도록 설정합니다.
        param.requires_grad = False

# 전략 2: pooler를 제외한 BERT 본체를 고정합니다.
elif tl_strategy == 2:
    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # pooler가 아닌 파라미터만 고정합니다.
        if not name.startswith("pooler"):
            # 해당 파라미터의 gradient 계산을 끕니다.
            param.requires_grad = False

# 전략 3: 마지막 Encoder Layer와 pooler만 학습합니다.
elif tl_strategy == 3:
    # BERT Base 구조의 마지막 encoder layer 이름을 지정합니다.
    last_layer_name = "layer.11"

    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # pooler도 아니고 마지막 layer도 아니면 고정합니다.
        if (not name.startswith("pooler")) and (last_layer_name not in name):
            # 해당 파라미터가 학습되지 않도록 설정합니다.
            param.requires_grad = False

# 학습 가능한 파라미터 수를 계산합니다.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# 전체 파라미터 수를 계산합니다.
total_params = sum(p.numel() for p in model.parameters())

# 학습 가능한 파라미터 비율을 출력합니다.
print(f"학습 가능 파라미터: {trainable_params:,} / 전체 파라미터: {total_params:,}")

학습 가능 파라미터: 7,683,847 / 전체 파라미터: 118,302,727


## 11. 하이퍼파라미터와 Optimizer 설정

 Epoch, Batch Size, Weight Decay 등의 설정을 PyTorch 학습 루프에 맞게 구성합니다.

In [71]:
# 전체 학습 Epoch 수를 지정합니다.
EPOCHS = 1

# BERT Fine-tuning에 사용할 학습률을 지정합니다.
LEARNING_RATE = 2e-5

# Weight Decay 정규화 계수를 지정합니다.
WEIGHT_DECAY = 0.01

# Warmup 단계 수를 지정합니다.
WARMUP_STEPS = 0

# 학습 가능한 파라미터만 AdamW Optimizer에 전달합니다.
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), # 고정되지 않은 파라미터만 업데이트합니다.
    lr=LEARNING_RATE,                                      # 학습률입니다.
    weight_decay=WEIGHT_DECAY                              # Weight Decay 계수입니다.
)

# 전체 학습 Step 수를 계산합니다.
total_training_steps = len(train_loader) * EPOCHS

# 선형 학습률 스케줄러를 생성합니다.
scheduler = get_linear_schedule_with_warmup(
    optimizer,                              # 학습률을 조정할 Optimizer입니다.
    num_warmup_steps=WARMUP_STEPS,          # Warmup 단계 수입니다.
    num_training_steps=total_training_steps # 전체 학습 Step 수입니다.
)

## 12. 학습 함수와 평가 함수 정의

다중 분류에서도 BERT 모델은 `labels`를 받으면 내부적으로 CrossEntropyLoss를 계산합니다.

In [72]:
# 한 Epoch 동안 모델을 학습하는 함수를 정의합니다.
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    # 모델을 학습 모드로 전환합니다.
    model.train()

    # 전체 loss를 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 실제 정답 레이블을 저장할 리스트를 생성합니다.
    all_labels = []

    # 예측 레이블을 저장할 리스트를 생성합니다.
    all_preds = []

    # 학습 DataLoader에서 배치를 하나씩 가져옵니다.
    for batch in tqdm(data_loader, desc="Training"):
        # input_ids를 학습 장치로 이동합니다.
        input_ids = batch["input_ids"].to(device)

        # attention_mask를 학습 장치로 이동합니다.
        attention_mask = batch["attention_mask"].to(device)

        # labels를 학습 장치로 이동합니다.
        labels = batch["labels"].to(device)

        # 이전 gradient를 초기화합니다.
        optimizer.zero_grad()

        # 모델에 입력을 전달하여 loss와 logits를 계산합니다.
        outputs = model(
            input_ids=input_ids,             # 뉴스 본문 토큰 ID입니다.
            attention_mask=attention_mask,   # 패딩 위치를 구분하는 마스크입니다.
            labels=labels                    # 정답 카테고리입니다.
        )

        # 모델이 계산한 손실값을 가져옵니다.
        loss = outputs.loss

        # 역전파로 gradient를 계산합니다.
        loss.backward()

        # gradient 폭주를 방지하기 위해 gradient norm을 제한합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Optimizer가 파라미터를 업데이트합니다.
        optimizer.step()

        # Scheduler가 학습률을 갱신합니다.
        scheduler.step()

        # 배치 loss를 누적합니다.
        total_loss += loss.item()

        # logits에서 가장 큰 값을 가진 클래스 인덱스를 예측값으로 선택합니다.
        preds = torch.argmax(outputs.logits, dim=1)

        # 정답 레이블을 CPU 리스트로 변환하여 누적합니다.
        all_labels.extend(labels.detach().cpu().numpy())

        # 예측 레이블을 CPU 리스트로 변환하여 누적합니다.
        all_preds.extend(preds.detach().cpu().numpy())

    # 평균 loss를 계산합니다.
    avg_loss = total_loss / len(data_loader)

    # 정확도를 계산합니다.
    accuracy = accuracy_score(all_labels, all_preds)

    # 평균 loss와 정확도를 반환합니다.
    return avg_loss, accuracy

# 모델 평가 함수를 정의합니다.
def evaluate(model, data_loader, device):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 전체 loss를 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 실제 정답 레이블을 저장할 리스트를 생성합니다.
    all_labels = []

    # 예측 레이블을 저장할 리스트를 생성합니다.
    all_preds = []

    # 평가 중에는 gradient 계산을 하지 않습니다.
    with torch.no_grad():
        # 평가 DataLoader에서 배치를 하나씩 가져옵니다.
        for batch in tqdm(data_loader, desc="Evaluating"):
            # input_ids를 평가 장치로 이동합니다.
            input_ids = batch["input_ids"].to(device)

            # attention_mask를 평가 장치로 이동합니다.
            attention_mask = batch["attention_mask"].to(device)

            # labels를 평가 장치로 이동합니다.
            labels = batch["labels"].to(device)

            # 모델에 입력을 전달하여 loss와 logits를 계산합니다.
            outputs = model(
                input_ids=input_ids,           # 뉴스 본문 토큰 ID입니다.
                attention_mask=attention_mask, # 패딩 위치를 구분하는 마스크입니다.
                labels=labels                  # 정답 카테고리입니다.
            )

            # 배치 loss를 누적합니다.
            total_loss += outputs.loss.item()

            # 가장 높은 logit을 가진 클래스를 예측값으로 선택합니다.
            preds = torch.argmax(outputs.logits, dim=1)

            # 정답 레이블을 CPU 리스트로 변환하여 누적합니다.
            all_labels.extend(labels.detach().cpu().numpy())

            # 예측 레이블을 CPU 리스트로 변환하여 누적합니다.
            all_preds.extend(preds.detach().cpu().numpy())

    # 평균 loss를 계산합니다.
    avg_loss = total_loss / len(data_loader)

    # 정확도를 계산합니다.
    accuracy = accuracy_score(all_labels, all_preds)

    # 평균 loss, 정확도, 전체 정답, 전체 예측을 반환합니다.
    return avg_loss, accuracy, all_labels, all_preds

In [73]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [74]:
print(dataset["category"].unique())
print(dataset["category"].min())
print(dataset["category"].max())

[6 5 3 2 1 0 4]
0
6


In [75]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(42000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [76]:
# print(dataset["소분류"].value_counts())

## 13. 모델 학습

Validation 성능을 확인하면서 뉴스 카테고리 분류 모델을 학습합니다.

In [ ]:
# 지정한 Epoch 수만큼 학습을 반복합니다.
for epoch in range(EPOCHS):
    # 현재 Epoch 번호를 출력합니다.
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # Train 데이터로 한 Epoch 학습합니다.
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)

    # Validation 데이터로 모델 성능을 평가합니다.
    valid_loss, valid_acc, _, _ = evaluate(model, valid_loader, device)

    # Train 손실과 정확도를 출력합니다.
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")

    # Validation 손실과 정확도를 출력합니다.
    print(f"Valid Loss: {valid_loss:.4f} | Valid Accuracy: {valid_acc:.4f}")


Epoch 1/1


Training:   0%|          | 0/514 [00:00<?, ?it/s]

## 14. Test 데이터 평가

학습이 끝난 후 Test 데이터로 최종 성능을 평가합니다.

In [ ]:
# Test 데이터로 최종 평가를 수행합니다.
test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader, device)

# Test 손실을 출력합니다.
print(f"Test Loss: {test_loss:.4f}")

# Test 정확도를 출력합니다.
print(f"Test Accuracy: {test_acc:.4f}")

# 카테고리 이름을 문자열로 변환합니다.
target_names = [str(label) for label in label_values]

# 클래스별 정밀도, 재현율, F1-score를 출력합니다.
print(classification_report(
    test_labels,              # 실제 카테고리 레이블입니다.
    test_preds,               # 모델 예측 카테고리 레이블입니다.
    target_names=target_names # 출력에 사용할 클래스 이름입니다.
))

## 15. 새 뉴스 문장 카테고리 예측 함수

학습된 BERT 모델에 새 뉴스 문장을 입력하여 카테고리를 예측합니다.

In [ ]:
# 새 뉴스 문장 하나를 입력받아 카테고리를 예측하는 함수를 정의합니다.
def predict_category(text, model, tokenizer, device, max_len=128):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 입력 뉴스 문장을 BERT 입력 형식으로 변환합니다.
    # encoded = tokenizer.encode_plus(
    encoded = tokenizer(
        text,                          # 예측할 뉴스 본문입니다.
        add_special_tokens=True,        # [CLS], [SEP] 토큰을 추가합니다.
        max_length=max_len,             # 최대 토큰 길이를 지정합니다.
        padding="max_length",          # 짧은 문장은 패딩합니다.
        truncation=True,                # 긴 문장은 자릅니다.
        return_attention_mask=True,     # attention_mask를 반환합니다.
        return_tensors="pt"             # PyTorch Tensor로 반환합니다.
    )

    # input_ids를 학습 장치로 이동합니다.
    input_ids = encoded["input_ids"].to(device)

    # attention_mask를 학습 장치로 이동합니다.
    attention_mask = encoded["attention_mask"].to(device)

    # 예측 과정에서는 gradient 계산을 하지 않습니다.
    with torch.no_grad():
        # 모델에 입력을 전달하여 logits를 계산합니다.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # logits를 softmax 확률로 변환합니다.
        probabilities = torch.softmax(outputs.logits, dim=1)

        # 가장 확률이 높은 클래스의 위치를 구합니다.
        predicted_index = torch.argmax(probabilities, dim=1).item()

    # 예측 인덱스를 실제 라벨 값으로 변환합니다.
    predicted_label = label_values[predicted_index]

    # 예측 라벨과 클래스별 확률을 반환합니다.
    return predicted_label, probabilities.squeeze(0).detach().cpu().numpy()

# 예측에 사용할 예시 뉴스 문장을 지정합니다.
example_news = "올림픽 대표팀이 결승전에서 승리하며 금메달을 획득했다."

# 예시 뉴스의 카테고리를 예측합니다.
pred_label, probs = predict_category(example_news, model, tokenizer, device, MAX_LEN)

# 입력 문장을 출력합니다.
print("입력 뉴스:", example_news)

# 예측 카테고리를 출력합니다.
print("예측 카테고리:", pred_label)

# 클래스별 확률을 출력합니다.
print("클래스별 확률:", probs)